# ⚡ Colab Media Studio — 云端 GPU 引擎

**使用说明**：
1. 菜单栏选择 **代码执行程序 → 更改运行时类型**：推荐 **T4**（测试/轻量）或 **L4**（性价比最高，推荐跑 Wan 2.2 5B）。
2. 依次运行下方 **第 1 步** 和 **第 2 步**。
3. 复制第 2 步输出的 **一键连接码**，粘贴到你 Mac 本地打开的 `Colab Media Studio` 即可像用本地显卡一样生图、生视频。
4. 用完请点本地 App 右上角红色 **「释放算力」**（或运行第 3 步），立刻停止扣减计算单元！


In [ ]:
#@title 1. 挂载 Google Drive 缓存 & 安装运行环境（首次约 1-2 分钟）
import os, subprocess, torch

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}

drive_mounted = False
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_mounted = True
        BASE_DIR = "/content/drive/MyDrive/ColabMediaStudio"
        print("✅ 已挂载 Google Drive，模型权重和作品将自动缓存在你的 Google One 云盘中（省时间=省计算单元）")
    except Exception as e:
        print(f"⚠️ 未挂载 Google Drive（{e}），将使用临时本地盘")
        BASE_DIR = "/content/ColabMediaStudio"
else:
    BASE_DIR = "/content/ColabMediaStudio"

HF_CACHE_DIR = os.path.join(BASE_DIR, "hf_cache")
OUTPUT_BACKUP_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_BACKUP_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["COLAB_STUDIO_DRIVE_MOUNTED"] = "1" if drive_mounted else "0"
os.environ["COLAB_STUDIO_OUTPUT_DIR"] = OUTPUT_BACKUP_DIR

assert torch.cuda.is_available(), "❌ 当前未分配到 GPU！请在顶部菜单选择：代码执行程序 -> 更改运行时类型 -> 选择 T4 / L4 / A100"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"🚀 当前 GPU: {gpu_name} ({vram_gb:.1f} GB 显存)")

print("📦 正在安装依赖（diffusers 最新版 + fastapi + cloudflared）...")
subprocess.check_call([
    "pip", "install", "-q", "-U",
    "fastapi", "uvicorn", "python-multipart", "pillow",
    "imageio", "imageio-ffmpeg", "hf_transfer",
    "accelerate", "transformers", "sentencepiece", "protobuf", "psutil"
])
subprocess.check_call([
    "pip", "install", "-q", "-U",
    "git+https://github.com/huggingface/diffusers.git"
])

if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"
    ])
    os.chmod("/usr/local/bin/cloudflared", 0o755)

print("✅ 环境准备就绪！请继续运行下方第 2 步启动云端服务。")


In [ ]:
#@title 2. 启动云端 GPU 引擎 & 生成本地 App 一键连接码（保持此格运行）
import base64, gc, inspect, io, os, re, secrets, shutil, subprocess, threading, time, traceback, uuid
from pathlib import Path
from typing import Any, Dict, Optional

import psutil
import torch
import uvicorn
from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.responses import FileResponse
from PIL import Image
from pydantic import BaseModel

# ------------------------------------------------------------
# Security token & telemetry state
# ------------------------------------------------------------
API_TOKEN = secrets.token_urlsafe(18)
START_TIME = time.time()
LAST_ACTIVITY = time.time()
DRIVE_MOUNTED = os.environ.get("COLAB_STUDIO_DRIVE_MOUNTED") == "1"
OUTPUT_BACKUP_DIR = Path(os.environ.get("COLAB_STUDIO_OUTPUT_DIR", "/content/ColabMediaStudio/outputs"))
LOCAL_TMP_DIR = Path("/content/studio_tmp_outputs")
LOCAL_TMP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_TOTAL_GB = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1)

def estimate_cu_rate(name: str, vram: float) -> float:
    u = name.upper()
    if "H100" in u:
        return 18.0
    if "A100" in u:
        return 9.5 if vram > 60 else 6.0
    if "V100" in u:
        return 3.5
    if "L4" in u:
        return 2.2
    if "T4" in u:
        return 1.2
    return 1.5

EST_CU_PER_HOUR = estimate_cu_rate(GPU_NAME, VRAM_TOTAL_GB)

STATE_LOCK = threading.Lock()
GPU_LOCK = threading.Lock()
LOADED_MODEL_KEY: Optional[str] = None
LOADED_PIPE: Any = None
ACTIVE_JOB_ID: Optional[str] = None
JOBS: Dict[str, Dict[str, Any]] = {}

def verify_token(authorization: Optional[str] = Header(default=None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing Bearer token")
    if authorization.split(" ", 1)[1].strip() != API_TOKEN:
        raise HTTPException(status_code=403, detail="Invalid token")

def unload_current_model():
    global LOADED_MODEL_KEY, LOADED_PIPE
    if LOADED_PIPE is not None:
        try:
            del LOADED_PIPE
        except Exception:
            pass
    LOADED_PIPE = None
    LOADED_MODEL_KEY = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def align_dim(val: int, mult: int = 32, minimum: int = 256) -> int:
    return max(minimum, int(round(val / mult)) * mult)

def align_frames(val: int, step: int = 8, minimum: int = 9) -> int:
    k = max(1, round((val - 1) / step))
    return max(minimum, k * step + 1)

def get_pipeline(model_id: str, has_init_image: bool, update_cb):
    global LOADED_MODEL_KEY, LOADED_PIPE
    cache_key = f"{model_id}:{'i2v' if has_init_image else 't2v'}"
    if LOADED_MODEL_KEY == cache_key and LOADED_PIPE is not None:
        return LOADED_PIPE

    unload_current_model()
    update_cb("loading_model", f"正在加载/下载模型 {model_id}（首次需下载至 Drive 缓存，请耐心等待）...", 8)
    low_vram = VRAM_TOTAL_GB <= 18.5

    if model_id == "flux-schnell":
        from diffusers import FluxPipeline
        pipe = FluxPipeline.from_pretrained(
            "black-forest-labs/FLUX.1-schnell",
            torch_dtype=torch.bfloat16,
        )
        if low_vram:
            pipe.enable_sequential_cpu_offload()
        else:
            pipe.enable_model_cpu_offload()
        if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
            pipe.vae.enable_tiling()

    elif model_id == "sdxl-base":
        from diffusers import StableDiffusionXLPipeline
        pipe = StableDiffusionXLPipeline.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
        )
        pipe.enable_model_cpu_offload()
        pipe.enable_vae_tiling()

    elif model_id == "ltx-video":
        if has_init_image:
            from diffusers import LTXImageToVideoPipeline
            pipe = LTXImageToVideoPipeline.from_pretrained(
                "Lightricks/LTX-Video",
                torch_dtype=torch.bfloat16,
            )
        else:
            from diffusers import LTXVideoPipeline
            pipe = LTXVideoPipeline.from_pretrained(
                "Lightricks/LTX-Video",
                torch_dtype=torch.bfloat16,
            )
        if low_vram:
            pipe.enable_sequential_cpu_offload()
        else:
            pipe.enable_model_cpu_offload()
        if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
            pipe.vae.enable_tiling()

    elif model_id == "wan2.1-1.3b":
        from diffusers import AutoencoderKLWan, WanPipeline
        repo = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
        vae = AutoencoderKLWan.from_pretrained(repo, subfolder="vae", torch_dtype=torch.float32)
        pipe = WanPipeline.from_pretrained(repo, vae=vae, torch_dtype=torch.bfloat16)
        pipe.enable_model_cpu_offload()

    elif model_id == "wan2.2-5b":
        from diffusers import AutoencoderKLWan, WanPipeline
        repo = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
        vae = AutoencoderKLWan.from_pretrained(repo, subfolder="vae", torch_dtype=torch.float32)
        pipe = WanPipeline.from_pretrained(repo, vae=vae, torch_dtype=torch.bfloat16)
        if VRAM_TOTAL_GB < 22:
            pipe.enable_sequential_cpu_offload()
        else:
            pipe.enable_model_cpu_offload()
    else:
        raise ValueError(f"不支持的模型: {model_id}")

    LOADED_MODEL_KEY = cache_key
    LOADED_PIPE = pipe
    return pipe

class GenerateRequest(BaseModel):
    task: str = "image"
    model_id: str = "flux-schnell"
    prompt: str
    negative_prompt: str = ""
    width: int = 768
    height: int = 512
    steps: int = 20
    guidance_scale: float = 3.5
    num_frames: int = 49
    fps: int = 24
    seed: int = -1
    init_image_base64: Optional[str] = None

def run_generation_job(job_id: str, req: GenerateRequest):
    global ACTIVE_JOB_ID, LAST_ACTIVITY
    t0 = time.time()

    def update(status: str, stage: str, progress: int, **extra):
        with STATE_LOCK:
            JOBS[job_id].update({
                "status": status,
                "stage": stage,
                "progress": progress,
                "elapsed_seconds": round(time.time() - t0, 1),
                **extra,
            })

    with GPU_LOCK:
        ACTIVE_JOB_ID = job_id
        LAST_ACTIVITY = time.time()
        try:
            init_img = None
            if req.init_image_base64:
                raw = base64.b64decode(req.init_image_base64)
                init_img = Image.open(io.BytesIO(raw)).convert("RGB")

            pipe = get_pipeline(req.model_id, init_img is not None, update)
            seed = req.seed if req.seed >= 0 else secrets.randbelow(2**31 - 1)
            gen = torch.Generator(device="cpu").manual_seed(seed)

            width = align_dim(req.width, 32)
            height = align_dim(req.height, 32)
            total_steps = max(1, int(req.steps))

            def step_callback(_pipe, step_idx, _timestep, callback_kwargs):
                pct = 15 + int(((step_idx + 1) / total_steps) * 78)
                update("running", f"GPU 正在推理中 ({step_idx + 1}/{total_steps} 步)...", min(93, pct), step=step_idx + 1, total_steps=total_steps)
                return callback_kwargs

            update("running", f"开始推理 (seed={seed})...", 15, seed=seed, total_steps=total_steps)

            if req.task == "image":
                kwargs = dict(
                    prompt=req.prompt,
                    width=width,
                    height=height,
                    num_inference_steps=total_steps,
                    guidance_scale=req.guidance_scale,
                    generator=gen,
                    callback_on_step_end=step_callback,
                )
                if req.model_id == "sdxl-base" and req.negative_prompt:
                    kwargs["negative_prompt"] = req.negative_prompt
                result = pipe(**kwargs)
                img = result.images[0]
                out_file = LOCAL_TMP_DIR / f"{job_id}.png"
                img.save(out_file)
                ext = "png"
            else:
                from diffusers.utils import export_to_video
                step_mult = 8 if req.model_id == "ltx-video" else 4
                num_frames = align_frames(req.num_frames, step=step_mult, minimum=9)
                if init_img is not None:
                    init_img = init_img.resize((width, height), Image.LANCZOS)

                call_params = inspect.signature(pipe.__call__).parameters
                kwargs = dict(
                    prompt=req.prompt,
                    width=width,
                    height=height,
                    num_frames=num_frames,
                    num_inference_steps=total_steps,
                    guidance_scale=req.guidance_scale,
                    generator=gen,
                )
                if "negative_prompt" in call_params and req.negative_prompt:
                    kwargs["negative_prompt"] = req.negative_prompt
                if "callback_on_step_end" in call_params:
                    kwargs["callback_on_step_end"] = step_callback
                if init_img is not None:
                    if "image" in call_params:
                        kwargs["image"] = init_img
                    elif req.model_id == "wan2.2-5b":
                        from diffusers import WanImageToVideoPipeline
                        i2v_pipe = WanImageToVideoPipeline.from_pipe(pipe)
                        kwargs["image"] = init_img
                        pipe = i2v_pipe

                frames = pipe(**kwargs).frames[0]
                update("encoding", "正在封装 MP4 视频...", 95)
                out_file = LOCAL_TMP_DIR / f"{job_id}.mp4"
                export_to_video(frames, str(out_file), fps=req.fps)
                ext = "mp4"

            drive_path = None
            if DRIVE_MOUNTED:
                try:
                    dst = OUTPUT_BACKUP_DIR / f"{int(time.time())}_{req.model_id}_{job_id[:8]}.{ext}"
                    shutil.copy2(out_file, dst)
                    drive_path = str(dst)
                except Exception:
                    pass

            update(
                "completed",
                "生成完成，正在传输到本地 App...",
                100,
                file_path=str(out_file),
                file_ext=ext,
                seed=seed,
                gpu_name=GPU_NAME,
                drive_path=drive_path,
            )
        except Exception as e:
            traceback.print_exc()
            update("failed", f"生成出错: {e}", 0, error=str(e))
        finally:
            ACTIVE_JOB_ID = None
            LAST_ACTIVITY = time.time()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

app = FastAPI(title="Colab Media Studio Engine")

@app.get("/api/health", dependencies=[Depends(verify_token)])
def health():
    now = time.time()
    uptime = now - START_TIME
    vram_used = round(torch.cuda.memory_allocated(0) / (1024 ** 3), 2) if torch.cuda.is_available() else 0.0
    vm = psutil.virtual_memory()
    return {
        "ok": True,
        "gpu_name": GPU_NAME,
        "vram_total_gb": VRAM_TOTAL_GB,
        "vram_used_gb": vram_used,
        "ram_total_gb": round(vm.total / (1024 ** 3), 1),
        "ram_used_gb": round(vm.used / (1024 ** 3), 1),
        "uptime_seconds": round(uptime, 1),
        "est_cu_per_hour": EST_CU_PER_HOUR,
        "est_cu_used": round((uptime / 3600.0) * EST_CU_PER_HOUR, 3),
        "drive_mounted": DRIVE_MOUNTED,
        "loaded_model": LOADED_MODEL_KEY,
        "active_job_id": ACTIVE_JOB_ID,
        "idle_seconds": 0 if ACTIVE_JOB_ID else round(now - LAST_ACTIVITY, 1),
    }

@app.post("/api/generate", dependencies=[Depends(verify_token)])
def generate(req: GenerateRequest):
    if ACTIVE_JOB_ID is not None:
        raise HTTPException(status_code=409, detail="云端 GPU 正在执行另一个任务，请稍候完成后再提交。")
    job_id = uuid.uuid4().hex
    with STATE_LOCK:
        JOBS[job_id] = {
            "job_id": job_id,
            "status": "queued",
            "stage": "任务已进入 GPU 队列...",
            "progress": 2,
            "elapsed_seconds": 0.0,
        }
    threading.Thread(target=run_generation_job, args=(job_id, req), daemon=True).start()
    return {"job_id": job_id, "status": "queued"}

@app.get("/api/jobs/{job_id}", dependencies=[Depends(verify_token)])
def job_status(job_id: str):
    with STATE_LOCK:
        job = JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    return {k: v for k, v in job.items() if k != "file_path"}

@app.get("/api/jobs/{job_id}/result", dependencies=[Depends(verify_token)])
def job_result(job_id: str):
    with STATE_LOCK:
        job = JOBS.get(job_id)
    if not job or job.get("status") != "completed":
        raise HTTPException(status_code=404, detail="Result not ready")
    path = job.get("file_path")
    if not path or not os.path.exists(path):
        raise HTTPException(status_code=404, detail="Output file missing")
    media_type = "video/mp4" if path.endswith(".mp4") else "image/png"
    return FileResponse(path, media_type=media_type)

@app.post("/api/unload", dependencies=[Depends(verify_token)])
def unload():
    if ACTIVE_JOB_ID is not None:
        raise HTTPException(status_code=409, detail="当前有任务正在运行，无法清理显存。")
    unload_current_model()
    return {"ok": True}

@app.post("/api/shutdown", dependencies=[Depends(verify_token)])
def shutdown():
    def _unassign():
        time.sleep(1.5)
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as e:
            print("自动释放失败，请手动断开运行时:", e)
    threading.Thread(target=_unassign, daemon=True).start()
    return {"ok": True, "message": "Colab runtime unassign scheduled"}

# Start Cloudflare Quick Tunnel & run Uvicorn inside this Colab kernel
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
deadline = time.time() + 40
while time.time() < deadline:
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

assert public_url, "❌ Cloudflare 隧道启动超时，请重新运行此单元格。"

print("\n" + "=" * 72)
print("🎉 Colab Media Studio 云端 GPU 引擎已上线！")
print(f"🖥️  GPU: {GPU_NAME} ({VRAM_TOTAL_GB} GB) | 预计费率: ~{EST_CU_PER_HOUR} 单元/小时")
print("📋 请复制下面整行【一键连接码】，粘贴到本地 Mac App 的「连接设置」中：")
print()
print(f"{public_url}::{API_TOKEN}")
print()
print("💡 用完后可直接在本机 App 点右上角红色「释放算力」自动停机止损！")
print("=" * 72 + "\n")

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
await server.serve()


In [ ]:
#@title 3. 手动紧急停机：立即断开并删除运行时（停止消耗计算单元）
from google.colab import runtime
print("正在释放 Colab 运行时并停止计算单元扣费...")
runtime.unassign()
